In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1. Localizar los archivos generados
output_base = Path("D:/TUSZ_DataLake/04_TUSZ_Features_ML/version=v1_raw_labels")
parquet_files = list(output_base.rglob("*.parquet"))

if not parquet_files:
    print(f"❌ No se encontraron archivos en {output_base}. ¿Corriste el piloto?")
else:
    # Tomamos el primero para la inspección
    sample_file = parquet_files[0]
    print(f"--- 📄 Auditando archivo: {sample_file.name} ---")
    
    df = pd.read_parquet(sample_file)
    
    # --- PRUEBAS DE CALIDAD ---
    
    # A. Verificación de Dimensiones
    # Deberían ser ~1143 columnas de features + metadata (start_time, split, label)
    print(f"\n1. Dimensiones (Filas, Columnas): {df.shape}")
    
    # B. Verificación de Columnas Críticas
    cols = df.columns.tolist()
    metadata_cols = ['start_time', 'label']
    missing_meta = [c for c in metadata_cols if c not in cols]
    
    if not missing_meta:
        print("✅ Columnas de metadata presentes.")
    else:
        print(f"⚠️ Faltan columnas de metadata: {missing_meta}")

    # C. Análisis de Características (Features)
    feature_cols = [c for c in cols if c not in metadata_cols]
    print(f"2. Total de características extraídas: {len(feature_cols)}")
    
    # D. Verificación del Escalado Robusto
    # Los valores deberían estar mayormente centrados cerca de 0
    stats = df[feature_cols[0]].describe()
    print(f"\n3. Estadística de la primera feature ({feature_cols[0]}):")
    print(f"   - Media: {stats['mean']:.4f}")
    print(f"   - Mediana: {df[feature_cols[0]].median():.4f}")
    
    # E. Distribución de Etiquetas
    print("\n4. Distribución de clases en este archivo:")
    print(df['label'].value_counts())
    
    # F. Integridad de Datos (Nulos)
    nan_count = df.isna().sum().sum()
    if nan_count == 0:
        print("\n✅ Cero valores nulos (NaN) detectados.")
    else:
        print(f"\n⚠️ Se detectaron {nan_count} valores nulos. Revisa el motor de cálculo.")

    print("\n--- Vista previa de las primeras filas ---")
    print(df[metadata_cols + feature_cols[:8]].head())

--- 📄 Auditando archivo: session_s002_2013.parquet ---

1. Dimensiones (Filas, Columnas): (326, 1086)
✅ Columnas de metadata presentes.
2. Total de características extraídas: 1084

3. Estadística de la primera feature (end_time):
   - Media: 654.0959
   - Mediana: 654.0960

4. Distribución de clases en este archivo:
label
bckg    326
Name: count, dtype: int64

✅ Cero valores nulos (NaN) detectados.

--- Vista previa de las primeras filas ---
   start_time label   end_time  spatial_ASI  spatial_APG  spatial_GFP  \
0         0.0  bckg   4.096000    -0.021417     0.048995     0.639324   
1         4.0  bckg   8.096000    -0.036507     0.013369     0.646234   
2         8.0  bckg  12.096000     0.034687    -0.005794     0.620551   
3        12.0  bckg  16.096001     0.005141     0.001821     0.628771   
4        16.0  bckg  20.096001    -0.112266     0.010504     0.658404   

   Fp1-F7_A5_rms  Fp1-F7_A5_pwr  Fp1-F7_A5_ll  Fp1-F7_A5_teo  
0       1.743310       3.039129      1.996593       

In [3]:
# Esto te dirá exactamente cuántos bits ocupa cada columna
print(df.dtypes.value_counts()) 
# O para ver una columna específica:
print(f"Tipo de dato de la señal: {df['Fp1-F7_A5_rms'].dtype}")

float32    1085
object        1
Name: count, dtype: int64
Tipo de dato de la señal: float32


In [2]:
df.head()

,start_time,end_time,label,spatial_ASI,spatial_APG,spatial_GFP,Fp1-F7_A5_rms,Fp1-F7_A5_pwr,Fp1-F7_A5_ll,Fp1-F7_A5_teo,...,Cz-Pz_D1_rms,Cz-Pz_D1_pwr,Cz-Pz_D1_ll,Cz-Pz_D1_teo,Cz-Pz_D1_mean,Cz-Pz_D1_std,Cz-Pz_D1_skew,Cz-Pz_D1_kurt,Cz-Pz_D1_ent,Cz-Pz_D1_dkatz
0,0.0,4.096000,bckg,-0.021417,0.048995,0.639324,1.743310,3.039129,1.996593,2.127595,...,0.126614,0.016031,0.193527,0.008204,0.000461,0.126613,0.001149,-0.350177,2.081704,-5.740728
1,4.0,8.096000,bckg,-0.036507,0.013369,0.646234,2.227133,4.960121,2.029925,3.051950,...,0.111895,0.012521,0.170566,0.006998,-0.000265,0.111895,-0.018409,0.106954,1.858826,-4.461590
2,8.0,12.096000,bckg,0.034687,-0.005794,0.620551,3.104839,9.640025,2.686069,7.700901,...,0.127504,0.016257,0.199293,0.008254,-0.000159,0.127504,-0.029196,-0.346056,1.928444,-5.095708
3,12.0,16.096001,bckg,0.005141,0.001821,0.628771,3.587301,12.868725,2.972025,11.633579,...,0.122349,0.014969,0.196354,0.006816,-0.000117,0.122349,0.029080,-0.537379,2.127985,-4.229201
4,16.0,20.096001,bckg,-0.112266,0.010504,0.658404,2.063335,4.257353,1.859327,4.423915,...,0.147975,0.021897,0.232844,0.009676,0.000175,0.147975,-0.032121,-0.272910,1.949147,-7.078492
